### **[Search a 2D Matrix (LeetCode 74)](https://leetcode.com/problems/search-a-2d-matrix/description/)**

This is a beautiful problem that tests your ability to abstract data structures. It asks you to look at a 2D grid but mentally process it as a simple 1D line.

Here is the comprehensive breakdown of how to tackle it, moving from brute force to the optimal logarithmic solution.

---

### **1. Constraint Analysis & Required Complexity**

* `m, n <= 100`: The matrix can be up to 100x100, meaning a maximum of 10,000 elements.
* **Time Complexity Requirement:** The problem explicitly demands an $O(\log(m \cdot n))$ time complexity.
* An $O(m \cdot n)$ brute-force approach would take 10,000 operations, which is easily handled by modern CPUs, but it fails the specific constraint of the problem.
* Any time you see $O(\log N)$ required on a **sorted** dataset, your brain should immediately scream: **Binary Search**.


* **Space Complexity Requirement:** The problem does not state a space requirement, but an optimal binary search requires only pointers, meaning we should aim for $O(1)$ constant space.

### **2. Core Concepts & Pattern**

* **Pattern:** Binary Search.
* **Core Concept:** 1D to 2D Index Mapping. Because the matrix guarantees that the last element of a row is smaller than the first element of the next row, the entire matrix is strictly sorted. If you were to flatten it out into a single row, it would just be one long, perfectly sorted array. We can perform a binary search on this "imaginary" flattened array.

### **3. Deep Dive: Binary Search & Index Mapping**

**Binary Search ($O(\log N)$):**
Instead of searching one by one, Binary Search looks at the middle element of a sorted list. If the target is smaller than the middle, it discards the right half. If it's larger, it discards the left half. It repeats this, cutting the search space in half every single time. This is why it is logarithmic—even with 1,000,000 items, it only takes at most 20 guesses to find the target.

**The Math of Flattening (Index Mapping):**
We want the speed of a 1D binary search, but flattening a 2D array in memory costs $O(m \cdot n)$ time and $O(m \cdot n)$ extra space. We don't want to actually flatten it; we want to *pretend* it's flattened.

If we have a matrix with `n` columns (width), we can convert any 1D index into a 2D `[row][col]` coordinate using simple division and modulo arithmetic:

* **Row:** `index // n` (Integer division tells us how many full rows we have bypassed).
* **Col:** `index % n` (Modulo gives us the remainder, which is our exact position in the current row).

### **4. The Intuition**

Let's look at `matrix = [[1,3,5,7], [10,11,16,20], [23,30,34,60]]` and `target = 3`.
The matrix has `m = 3` rows and `n = 4` columns. Total elements = 12.
Our imaginary 1D array has indices from `0` to `11`.

1. We set our `low` pointer to `0` and `high` pointer to `11`.
2. We find the `mid` index: `(0 + 11) // 2 = 5`.
3. We map index `5` back to 2D:
* `row = 5 // 4 = 1`
* `col = 5 % 4 = 1`


4. We look at `matrix[1][1]`, which is `11`.
5. `11` is greater than our target `3`. So, we discard the entire right half of our imaginary array by moving `high` down to `mid - 1` (which is `4`).
6. We repeat until we find the target or the pointers cross!

### **5. The Solutions**

#### **Solution 1: Brute Force Approach**

We just loop through every single element in the matrix. We ignore the sorted properties completely.

```python
class Solution:
    def searchMatrix(self, matrix: list[list[int]], target: int) -> bool:
        rows = len(matrix)
        cols = len(matrix[0])
        
        # Check every element one by one
        for r in range(rows):
            for c in range(cols):
                if matrix[r][c] == target:
                    return True
                    
        return False

```

* **Time Complexity:** $O(m \cdot n)$. We check every element.
* **Space Complexity:** $O(1)$.

---

#### **Solution 2: Better Approach (Double Binary Search)**

We take advantage of the sorting. First, we binary search the **first column** of each row to find which row the target *might* belong to. Once we find the correct row, we run a second binary search specifically on that single row.

```python
class Solution:
    def searchMatrix(self, matrix: list[list[int]], target: int) -> bool:
        rows = len(matrix)
        cols = len(matrix[0])
        
        # 1. Binary Search to find the correct row
        top, bot = 0, rows - 1
        while top <= bot:
            row = (top + bot) // 2
            if target > matrix[row][-1]:
                top = row + 1
            elif target < matrix[row][0]:
                bot = row - 1
            else:
                break # We found the row where the target should be!
                
        # If top crossed bot, the target isn't in the range of the matrix
        if not (top <= bot):
            return False
            
        # 2. Binary Search within the specific row
        row = (top + bot) // 2
        left, right = 0, cols - 1
        while left <= right:
            mid = (left + right) // 2
            if target > matrix[row][mid]:
                left = mid + 1
            elif target < matrix[row][mid]:
                right = mid - 1
            else:
                return True
                
        return False

```

* **Time Complexity:** $O(\log m + \log n)$, which mathematically simplifies to $O(\log(m \cdot n))$.
* **Space Complexity:** $O(1)$.

---

#### **Solution 3: Optimal Approach (Single Imaginary 1D Binary Search)**

This is the cleanest and most mathematically elegant solution. We treat the 2D matrix exactly like a 1D array using our index mapping concept.

```python
class Solution:
    def searchMatrix(self, matrix: list[list[int]], target: int) -> bool:
        if not matrix or not matrix[0]:
            return False
            
        rows = len(matrix)
        cols = len(matrix[0])
        
        # Set pointers for the "imaginary" flattened 1D array
        low = 0
        high = (rows * cols) - 1
        
        while low <= high:
            mid = (low + high) // 2
            
            # The Magic: Convert the 1D 'mid' index back into 2D coordinates
            mid_row = mid // cols
            mid_col = mid % cols
            mid_value = matrix[mid_row][mid_col]
            
            if mid_value == target:
                return True
            elif mid_value < target:
                low = mid + 1  # Target is in the right half
            else:
                high = mid - 1 # Target is in the left half
                
        return False

```

* **Time Complexity:** $O(\log(m \cdot n))$. We cut the entire dataset in half with every step.
* **Space Complexity:** $O(1)$. No extra arrays were created, purely using mathematical mapping.

To help solidify how `mid // cols` and `mid % cols` work together to navigate a 2D space seamlessly, you can step through this visualizer.